# README: Slack URL Summarizer & Tagger

## Overview
This notebook is designed to automate the process of extracting, summarizing, and tagging web links shared in Slack messages. It pulls unprocessed URLs from a source Airtable, scrapes the content, processes it using a local quantized Large Language Model (Qwen2.5-7B) along with keyword-matching algorithms, and writes the enriched data into a target Airtable.

## Steps Overview
1. **Environment Setup & Model Loading:** Installs required dependencies (Transformers, Trafilatura, etc.) and initializes the 4-bit quantized Qwen LLM.
2. **Airtable Configuration:** Establishes a connection to the Airtable API, reads the configuration secrets, and maps the input/output fields.
3. **Content Extraction:** Fetches the raw HTML/PDF content from the URLs using resilient scrapers, handling timeouts and basic parsing.
4. **Summarization & Tagging:** Applies a hybrid strategy:
    - **LLM:** Generates a concise summary, title, and key bullet points from the scraped text.
    - **Keyword Matching:** Assigns standardized tags (Resource, Concept, Topic, Sub-Topic) based on predefined vocabularies.
5. **Data Synchronization:** Writes the newly processed records to the target Airtable and exports a local CSV backup for auditing.

## Secrets Configuration
To run this notebook successfully, you will need to configure your Colab Secrets (the 🔑 icon on the left panel). The required variables, specifically `AIRTABLE_PAT` and `AIRTABLE_SLACK_BASE`, can be found securely stored in **Passbolt**.

## ⚠️ Future Industrialization Considerations
If this pipeline is to be deployed in a production/industrial environment, please note the following critical limitations and required improvements:

* **Web Access Restrictions:** Many websites currently restrict automated access (e.g., bot protection, paywalls, missing permissions). Consequently, the web scraper will fail to retrieve content, leading to the pipeline prompting `"[WARN] LLM failed for..."`.
* **Legal Access Configuration:** Future iterations must consider how to legally and reliably configure access permissions (e.g., using authenticated proxies, official APIs, or specialized scraping services that handle JavaScript/Captchas ethically).
* **Preventing Database Pollution:** Currently, when the LLM or scraper fails, the system generates an `[LLM-FALLBACK]` placeholder. In an industrial setup, the code must be refactored to handle these fallbacks strictly—either by flagging them for human review or dropping them entirely—to prevent default/fallback data from polluting the Airtable database.

In [ ]:
# ===== Install (once) =====
!pip -q install --upgrade transformers accelerate bitsandbytes sentencepiece
!pip -q install trafilatura pypdf beautifulsoup4 lxml html5lib tldextract
!pip -q install pandas tqdm requests python-dateutil


In [ ]:
# ===== Imports & config =====
import os, json, time, unicodedata, requests, tldextract, re
import pandas as pd
from datetime import datetime, timezone
from tqdm import tqdm
from bs4 import BeautifulSoup
import trafilatura
from pypdf import PdfReader
from io import BytesIO
import torch
from google.colab import userdata

# ===== LLM (Qwen2.5-7B-Instruct 4-bit) =====
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_NAME = os.environ.get("HF_MODEL", "Qwen/Qwen2.5-7B-Instruct")
tokenizer = None
model = None
MODEL_READY = False   # flag for initialization

def load_model():
    """Load 4-bit model once."""
    global tokenizer, model, MODEL_READY
    if MODEL_READY:
        return
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
    )
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="auto", quantization_config=bnb_cfg, trust_remote_code=True
    )
    mdl.eval()
    tokenizer, model, MODEL_READY = tok, mdl, True
    print("Model loaded:", MODEL_NAME)

load_model()

In [ ]:
# ============================================================
# Summarize & Tag Slack URLs → write into an EXISTING Airtable table
# - LLM for title / summary / key_points
# - Keyword-based tagging for Resource / Concept / Topic / Sub-Topic
# - Do NOT modify the source table
# - Deduplicate by URL; only process URLs not yet in target table
# ============================================================


# ===== Params (Colab UI) =====
TAG_STRATEGY = "hybrid_llm_title_keyword_tags"   # @param ["hybrid_llm_title_keyword_tags","keyword_based","llm_classification","embedding_distance","compare_all_methods"]
TOP_K_PER_FIELD = 5                    # @param {type:"integer"}
TOPIC_TOP1 = True                      # @param {type:"boolean"}
WRITE_TO_AIRTABLE = True               # @param {type:"boolean"}
LIMIT = 600                            # @param {type:"integer"}

# ---- Airtable secrets ----
AIRTABLE_TOKEN   = userdata.get("AIRTABLE_PAT")
AIRTABLE_BASE_ID = userdata.get("AIRTABLE_SLACK_BASE")
AIRTABLE_TABLE   = os.environ.get("AIRTABLE_SLACK_TABLE_NAME", "Slack_Messages")  # source table
AIRTABLE_API     = f"https://api.airtable.com/v0/{AIRTABLE_BASE_ID}/{AIRTABLE_TABLE}"
HEADERS_AT = {"Authorization": f"Bearer {AIRTABLE_TOKEN}", "Content-Type": "application/json"}

# ---- Target table (must already exist) ----
NEW_TABLE_NAME = os.environ.get("AIRTABLE_RESULT_TABLE", "Slack_Messages_LLM_Summaries")

# ---- Preferred field names in source table ----
PREF_FIELD_URL     = os.environ.get("FIELD_URL", "url")
PREF_FIELD_SUMMARY = os.environ.get("FIELD_SUMMARY", "summary")
PREF_FIELD_RES     = os.environ.get("FIELD_TAG_RESOURCE", "tag_resource")
PREF_FIELD_CON     = os.environ.get("FIELD_TAG_CONCEPT", "tag_concept")
PREF_FIELD_TOP     = os.environ.get("FIELD_TAG_TOPIC", "tag_topic")
PREF_FIELD_SUB     = os.environ.get("FIELD_TAG_SUBTOPIC", "tag_subtopic")
FIELD_TEXT_RAW     = None  # keep None; never write back to source

# ===== Airtable helpers =====
def airtable_request(method: str, url: str, **kwargs):
    """Robust Airtable request with 429 backoff and error surfacing."""
    max_retries, backoff = 5, 1.3
    for i in range(max_retries):
        r = requests.request(method, url, headers=HEADERS_AT, timeout=30, **kwargs)
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 1)) * (backoff ** i)
            time.sleep(wait); continue
        if r.ok:
            return r
        try:
            detail = r.json()
        except Exception:
            detail = r.text
        raise RuntimeError(f"Airtable {r.status_code}: {detail}")
    raise RuntimeError("Airtable: exceeded max retries")

def airtable_list_records(formula=None, page_size=100, max_pages=200):
    """List records from the SOURCE table."""
    records, offset = [], None
    for _ in range(max_pages):
        params = {"pageSize": page_size}
        if offset: params["offset"] = offset
        if formula: params["filterByFormula"] = formula
        data = airtable_request("GET", AIRTABLE_API, params=params).json()
        records.extend(data.get("records", []))
        offset = data.get("offset")
        if not offset: break
    return records

def airtable_list_records_from(table_name: str, fields: list[str] | None = None, page_size=100, max_pages=200):
    """List records from ANY table (used for the TARGET table)."""
    api = f"https://api.airtable.com/v0/{AIRTABLE_BASE_ID}/{table_name}"
    records, offset = [], None
    for _ in range(max_pages):
        params = {"pageSize": page_size}
        if offset: params["offset"] = offset
        if fields: params["fields[]"] = fields
        data = airtable_request("GET", api, params=params).json()
        records.extend(data.get("records", []))
        offset = data.get("offset")
        if not offset: break
    return records

# ===== HTML/PDF extraction =====
def extract_text(url: str, timeout=45) -> tuple[str, str]:
    """
    Fetch URL and extract main text.
    Returns: (text, content_type)
    """
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (LLM-summary-bot)"})
        r.raise_for_status()
        ctype = (r.headers.get("Content-Type") or "").lower()
        if "application/pdf" in ctype or url.lower().endswith(".pdf"):
            pdf = PdfReader(BytesIO(r.content))
            text = "\n".join([(p.extract_text() or "") for p in pdf.pages[:50]])
            return text, "application/pdf"
        txt = trafilatura.extract(
            r.text, include_comments=False,
            include_tables=False, favor_recall=True
        )
        if txt: return txt, (ctype or "text/html")
        soup = BeautifulSoup(r.text, "lxml")
        return soup.get_text(separator="\n"), (ctype or "text/html")
    except Exception:
        return "", ""

# ===== Tag dictionaries & clamp (with synonyms) =====
def _norm(s: str) -> str:
    s = s or ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    # keep letters, digits, spaces
    s = re.sub(r"[^a-z0-9\s\-_/+]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

ALLOWED = {
    "Concept": [
        "paradigme interface utilisateur","intention","noncommand user interface","heuristiques",
        "engagement","satisfaction","flexibilité","interaction naturelle","GenAI","modèles UX","interface humain-IA",
        "principe de conception","responsable","modèle mentaux","confiance","fiabilité","co-conception",
        "imperfection","variabilité générative","niveaux de risque","transparence","coherence","curation",
        "contexte social","UTAUT","Fogg Behavior Model","decouvrabilité","affordance","signifiants",
        "feedback","mapping","contraintes","modèle conceptuel","intuitif","utilisable","emotion",
        "ton de la voix","conception double-diamant","application","ethique","CxD","besoin utilisateur",
        "evolutive","stratégie commerciale","biais","design de conversation","consequences","Big Data",
        "prototypage","innovation","développement","sécurité","inclusif","savoir-faire","entretiens",
        "ergonomie","geste","artisanat","numériser","progression","design","méthode","expertise",
        "code","santé","modèles UX"
    ],
    "Resource": ["Article media","Article scientifique","Thèse","Site web","Vidéo","Podcast","Livre","Exposition"],
    "Topic": ["IA et dev","Management de startups et de produits","Design"],
    "Sub-Topic": [
        "Shape Up","Concept Maturity Level","Graphes de connaissance","Autres","Intelligence Artificielle",
        "Product Design","Ergonomie cognitive","Conception d’interfaces",
        "Programmation logique inductive (Inductive Logic Programming, ILP)",
        "Raisonnement basé sur les cas (Case-Based Reasoning, CBR)",
        "Lean Startup","Extreme Programming (XP)","Psychologie cognitive","Développement","User-centered Design",
        "Blue Ocean Strategy","Machine learning","Réseaux bayésiens",
        "Systèmes multi-agents (Multi-Agent Systems, MAS)","Product Management","Neurosciences cognitives",
        "Design Thinking","Savoir-faire","Business Model Canvas","Kanban","Usage Context-Based Design",
        "Discovery","Growth Hacking","Innovation","Jobs-To-Be-Done","Scrum","User Story Mapping",
        "Approche générale","Deep learning","Human-Centered Artificial Intelligence","DevOps",
        "Artificial Intelligence Project Cycle","Design Thinking for AI","Design Fiction",
        "Réglementation","IA générative","Lean AI","Ethique","Data-Driven Design",
        "Méthode OKR (Objectives Key Result)","Domain-driven Design","Lean UX","Customer Development",
        "Value-Sensitive Design","Responsible AI Design","Human-Centered Design"
    ]
}
ALLOWED_NORM = {k: [_norm(x) for x in v] for k,v in ALLOWED.items()}

SYNONYMS = {
    "gen ai":"GenAI","gen-ai":"GenAI","generative ai":"GenAI",
    "ux models":"modèles UX","ux":"modèles UX",
    "human-ai interface":"interface humain-IA","human ai interface":"interface humain-IA",
    "mental models":"modèle mentaux","discoverability":"decouvrabilité",
    "security":"sécurité","ethics":"ethique","trust":"confiance",
    "double diamond":"conception double-diamant","voice tone":"ton de la voix",
    "affordances":"affordance","feedbacks":"feedback","constraints":"contraintes",
    "okrs":"Méthode OKR (Objectives Key Result)","okr":"Méthode OKR (Objectives Key Result)",
    "ilp":"Programmation logique inductive (Inductive Logic Programming, ILP)",
    "cbr":"Raisonnement basé sur les cas (Case-Based Reasoning, CBR)",
    "mas":"Systèmes multi-agents (Multi-Agent Systems, MAS)",
}
def _synmap(label: str) -> str:
    return SYNONYMS.get(_norm(label), label)

def clamp_one(label: str, allowed_norm_list, allowed_raw_list):
    if not label: return None
    label = _synmap(label); n = _norm(label)
    if n in allowed_norm_list:
        return allowed_raw_list[allowed_norm_list.index(n)]
    hits = [i for i,a in enumerate(allowed_norm_list) if n in a or a in n]
    return allowed_raw_list[hits[0]] if hits else None

def clamp_tags(tag_dict: dict):
    """Clamp tags to allowed vocabularies; return dict with lists."""
    out = {"Resource":[], "Concept":[], "Topic":[], "Sub-Topic":[]}
    for raw in (tag_dict.get("Concept") or []):
        v = clamp_one(raw, ALLOWED_NORM["Concept"], ALLOWED["Concept"])
        if v and v not in out["Concept"]:
            out["Concept"].append(v)
    for k in ["Resource","Topic","Sub-Topic"]:
        got = None
        for raw in (tag_dict.get(k) or []):
            v = clamp_one(raw, ALLOWED_NORM[k], ALLOWED[k])
            if v: got = v; break
        if got: out[k] = [got]
    return out

# ===== Stopwords =====
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below between both but by
could did do does doing down during each few for from further had has have having he her here hers herself him himself his how i if in into is it its itself
let me more most my myself nor of on once only or other our ours ourselves out over own same she should so some such than that the their theirs them
themselves then there these they this those through to too under until up very was we were what when where which while who whom why with you your yours
yourself yourselves
un une des le la les de du au aux et ou mais donc or ni car à en pour sur par avec sans sous entre chez c d j l m n s t y qu que qui quoi dont où
""".split())

# ===== Source field auto-detect =====
def detect_fields():
    sample = airtable_list_records(formula=None, page_size=5, max_pages=1)
    if not sample:
        raise SystemExit("No records found. Check BASE/TABLE ids or permissions.")
    keys = list(sample[0].get("fields", {}).keys())

    def pick(pref: str, synonyms: list[str]):
        def canon(s):
            s = unicodedata.normalize("NFKD", s or "")
            s = "".join(ch for ch in s if not unicodedata.combining(ch))
            return s.lower().strip()
        keyset = {canon(k): k for k in keys}
        for c in [pref] + synonyms:
            if canon(c) in keyset:
                return keyset[canon(c)]
        return None

    url_f = pick(PREF_FIELD_URL, ["URL","Url","link","Link"])
    sum_f = pick(PREF_FIELD_SUMMARY, ["Summary","Résumé","Resume","abstract","synthese","synthesis"])
    res_f = pick(PREF_FIELD_RES, ["Resource","Type","Ressource","tag ressource"])
    con_f = pick(PREF_FIELD_CON, ["Concept","concepts","tags concept"])
    top_f = pick(PREF_FIELD_TOP, ["Topic","topic","domaine"])
    sub_f = pick(PREF_FIELD_SUB, ["Sub-Topic","Sub Topic","Sous-Topic","Sous-topic","Sous thème","Subtopic"])

    if url_f is None:
        raise SystemExit(f"Missing required fields in table: URL. Found fields: {keys}")

    return url_f, sum_f, res_f or PREF_FIELD_RES, con_f or PREF_FIELD_CON, top_f or PREF_FIELD_TOP, sub_f or PREF_FIELD_SUB, keys

FIELD_URL, FIELD_SUMMARY, FIELD_TAG_RESOURCE, FIELD_TAG_CONCEPT, FIELD_TAG_TOPIC, FIELD_TAG_SUBTOPIC, EXAMPLE_KEYS = detect_fields()
print("Resolved fields ->",
      {"url":FIELD_URL,"summary":FIELD_SUMMARY,"resource":FIELD_TAG_RESOURCE,"concept":FIELD_TAG_CONCEPT,"topic":FIELD_TAG_TOPIC,"subtopic":FIELD_TAG_SUBTOPIC})

def build_formula():
    """Fetch rows with URL present; if summary exists, prefer rows lacking summary."""
    if FIELD_SUMMARY is None:
        return f"AND(NOT({{{FIELD_URL}}}=''))"
    return f"AND(NOT({{{FIELD_URL}}}=''), OR({{{FIELD_SUMMARY}}}='', IS_BLANK({{{FIELD_SUMMARY}}})))"

# ===== Canonical URL & domain =====
def canonicalize_url(u: str) -> str:
    u = (u or "").strip()
    if not u:
        return ""
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", u):
        u = "https://" + u
    return u

def get_domain(u: str) -> str:
    u = canonicalize_url(u)
    ext = tldextract.extract(u)
    parts = [ext.subdomain, ext.domain, ext.suffix]
    dom = ".".join([p for p in parts if p])
    return dom.lower()

def system_prompt():
    return ("You are an expert research assistant. Read the provided web page text, "
            "then return a strict JSON with fields: title, source_domain, summary (English, <=180 words), "
            "key_points (3-7 bullet strings), and tags with keys: Concept (list), Resource (list, <=1), "
            "Topic (list, <=1), Sub-Topic (list, <=1). Use only labels from the allowed lists (case-insensitive). "
            "When unsure, leave that tag empty. Return JSON only.")

def build_user_msg(url: str, text: str, domain: str) -> str:
    """Truncate page text to control context size."""
    from transformers import AutoTokenizer as _AT
    tok = tokenizer if tokenizer else _AT.from_pretrained(MODEL_NAME, use_fast=True)
    max_tokens = 3200
    ids = tok(text, add_special_tokens=False, return_attention_mask=False)["input_ids"]
    cut = tok.decode(ids[:max_tokens]) if len(ids) > max_tokens else text
    return f"URL: {url}\nSource domain: {domain}\n\nPAGE_TEXT:\n{cut}\n"

def chat_json(messages, max_new_tokens=640, temperature=0.2):
    """Call chat model and parse JSON safely."""
    if not (model and tokenizer):
        raise SystemExit("Model not loaded. Call load_model().")
    chat = [{"role":"system","content":messages[0]}, {"role":"user","content":messages[1]}]
    input_ids = tokenizer.apply_chat_template(chat, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens,
            do_sample=(temperature>0), temperature=0.2,
            repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
    s, e = resp.find("{"), resp.rfind("}")
    if s!=-1 and e!=-1: resp = resp[s:e+1]
    try:
        return json.loads(resp)
    except Exception:
        return {"title":"","source_domain":"","summary":"","key_points":[],"tags":{}}

# ===== Tag fallbacks (ensure non-empty) =====
def infer_resource_from_ctype(ctype: str, url: str) -> str:
    c = (ctype or "").lower()
    if "pdf" in c: return "Article scientifique"
    if any(url.lower().endswith(x) for x in [".pdf"]): return "Article scientifique"
    return "Article media"

def ensure_minimum_tags(tags: dict, ctype: str, url: str) -> dict:
    """
    Guarantee at least some tags:
    - Resource: inferred from content-type (default 'Article media')
    - Topic: default 'Design'
    - Sub-Topic: default 'Approche générale'
    - Concept: default 'innovation'
    """
    out = {k: list(tags.get(k, [])) for k in ["Resource","Concept","Topic","Sub-Topic"]}
    if not out["Resource"]:
        r = clamp_one(infer_resource_from_ctype(ctype, url), ALLOWED_NORM["Resource"], ALLOWED["Resource"])
        if r: out["Resource"] = [r]
    if not out["Topic"]:
        out["Topic"] = ["Design"]
    if not out["Sub-Topic"]:
        out["Sub-Topic"] = ["Approche générale"]
    if not out["Concept"]:
        out["Concept"] = ["innovation"]
    return out

DEFAULT_PLACEHOLDERS = {
    "Resource": set(),                  # we don't treat Resource fallback as placeholder
    "Concept": {"innovation"},
    "Topic": {"Design"},
    "Sub-Topic": {"Approche générale"},
}

# ===== Method 1: keyword-based tagging from raw page text =====
def method1_tags_from_text(raw_text: str, url: str, ctype: str, topk: int = 5, topic_top1: bool = True) -> dict:
    """
    Normalize text → count occurrences of allowed labels' normalized forms (substring count).
    Pick top-k per field; Topic can be limited to top1.
    """
    text = _norm(raw_text)

    def score_field(allowed_raw, allowed_norm):
        scores = []
        for raw, norm in zip(allowed_raw, allowed_norm):
            if not norm:
                scores.append((raw, 0)); continue
            cnt = len(re.findall(r"(?<![a-z0-9])" + re.escape(norm) + r"(?![a-z0-9])", text))
            if cnt == 0 and " " in norm:
                toks = [t for t in norm.split() if t not in STOPWORDS]
                if toks:
                    cnt = sum(text.count(t) for t in toks)
            scores.append((raw, cnt))
        scores.sort(key=lambda x: x[1], reverse=True)
        return [lab for lab, c in scores if c > 0]

    res = [infer_resource_from_ctype(ctype, url)]  # single best guess for Resource
    concept_ranked   = score_field(ALLOWED["Concept"],   ALLOWED_NORM["Concept"])
    topic_ranked     = score_field(ALLOWED["Topic"],     ALLOWED_NORM["Topic"])
    subtopic_ranked  = score_field(ALLOWED["Sub-Topic"], ALLOWED_NORM["Sub-Topic"])

    concept = concept_ranked[:max(1, topk)] if concept_ranked else []
    topic   = topic_ranked[:1] if topic_top1 else topic_ranked[:max(1, topk)]
    subtop  = subtopic_ranked[:max(1, topk)] if subtopic_ranked else []

    out = {"Resource": res, "Concept": concept, "Topic": topic, "Sub-Topic": subtop}
    return ensure_minimum_tags(out, ctype, url)

# ===== Method 2: LLM-based tagging (backup) =====
def method2_tags_llm(raw_text: str, url: str, domain: str, ctype: str) -> dict:
    data = chat_json([system_prompt(), build_user_msg(url, raw_text, domain)])
    data.setdefault("tags", {})
    clamped = clamp_tags(data["tags"])
    return ensure_minimum_tags(clamped, ctype, url)

# ===== Method 3: Embedding distance (placeholder) =====
def method3_tags_embeddings(raw_text: str, url: str, ctype: str) -> dict:
    return ensure_minimum_tags({"Resource":[], "Concept":[], "Topic":[], "Sub-Topic":[]}, ctype, url)

# ===== Read processed URLs from TARGET table =====
def get_processed_urls(target_table: str) -> set[str]:
    try:
        recs = airtable_list_records_from(target_table, fields=["url"])
    except RuntimeError as e:
        raise RuntimeError(f"Cannot read target table '{target_table}'. Ensure it exists and token has access. {e}")
    urls = set()
    for r in recs:
        u = r.get("fields", {}).get("url")
        if isinstance(u, str) and u.strip():
            urls.add(canonicalize_url(u))
    return urls

# ===== Payload formatters & writer with auto-fallback =====
def _format_row_for_multiselect(row: dict) -> dict:
    """Assume tag fields are multi-selects (lowercase column names)."""
    out = dict(row)
    for k in ["resource","concept","topic","subtopic"]:
        vals = row.get(k, [])
        out[k] = [{"name": v} for v in vals]
    return out

def _format_row_for_text(row: dict) -> dict:
    """Assume tag fields are plain text; join with comma."""
    out = dict(row)
    for k in ["resource","concept","topic","subtopic"]:
        vals = row.get(k, [])
        out[k] = ", ".join(vals)
    return out

def write_records_existing_table(table_name: str, rows: list[dict], batch_size: int = 10):
    """
    Try POST with multi-select payload first.
    If Airtable returns 422 INVALID_VALUE_FOR_COLUMN, fallback to comma-joined strings
    and retry the SAME batch.
    """
    api = f"https://api.airtable.com/v0/{AIRTABLE_BASE_ID}/{table_name}"
    i = 0
    while i < len(rows):
        batch = rows[i:i+batch_size]

        # Attempt 1: multi-select
        payload_ms = {"records": [{"fields": _format_row_for_multiselect(r)} for r in batch], "typecast": True}
        try:
            airtable_request("POST", api, data=json.dumps(payload_ms))
            i += batch_size
            time.sleep(0.2)
            continue
        except RuntimeError as e:
            if "INVALID_VALUE_FOR_COLUMN" not in str(e):
                raise

        # Attempt 2: fallback to text
        payload_text = {"records": [{"fields": _format_row_for_text(r)} for r in batch], "typecast": True}
        airtable_request("POST", api, data=json.dumps(payload_text))
        i += batch_size
        time.sleep(0.2)

# ===== Stats helper (for method4) =====
def compute_stats(rows: list[dict], label: str = "", defaults: dict | None = None):
    """
    Compute, per field, the coverage rate (>=1 non-placeholder tag) and
    the average number of non-placeholder tags per row.
    """
    defaults = defaults or {}
    fields = ["resource","concept","topic","subtopic"]

    if not rows:
        return {
            "label": label, "n": 0,
            "has_Resource(%)": 0.0, "avg#Resource": 0.0,
            "has_Concept(%)": 0.0, "avg#Concept": 0.0,
            "has_Topic(%)": 0.0, "avg#Topic": 0.0,
            "has_Sub-Topic(%)": 0.0, "avg#Sub-Topic": 0.0,
        }

    n = len(rows)
    out = {"label": label, "n": n}

    for f in fields:
        ph = set(defaults.get(f, []))
        cover = 0
        total_vals = 0
        for r in rows:
            vals = list(r.get(f, []))
            clean = [v for v in vals if v not in ph]
            total_vals += len(clean)
            if len(clean) > 0:
                cover += 1
        out[f"has_{f}(%)"] = round(100.0 * cover / n, 2)
        out[f"avg#{f}"] = round(total_vals / n, 3)

    return out

# ===== Optional: SORT BY CREATED TIME =====
def extract_created_time(rec):
    f = rec.get("fields", {})
    for key in ["sent_at"]:
        if key in f:
            return f[key]
    return None


# ===== Main: HYBRID STRATEGY =====
def process_with_strategy(limit=60, strategy="hybrid_llm_title_keyword_tags", write_out=True, topk=5, topic_top1=True):
    """
    - Read candidates from source (filter formula)
    - Skip URLs already present in target
    - Extract text
    - HYBRID: LLM for title/summary/key_points, keyword-based for tags
    - Still keeps other strategies for compatibility if needed
    """
    # Need LLM for: llm_classification, compare_all_methods, hybrid_llm_title_keyword_tags
    need_llm = (strategy in ["llm_classification", "compare_all_methods", "hybrid_llm_title_keyword_tags"])
    # if need_llm and not MODEL_READY:
    #     load_model()

    formula = build_formula()
    src_records = airtable_list_records(formula=formula)
    print("Fetched records from source:", len(src_records))
    src_records.sort(key=lambda r: extract_created_time(r) or "", reverse=False)

    processed = get_processed_urls(NEW_TABLE_NAME)
    print("Already processed URLs in target:", len(processed))

    url_map = {}
    for rec in src_records:
        rid = rec.get("id")
        f = rec.get("fields", {})
        url = canonicalize_url((f.get(FIELD_URL) or "").strip())
        if not url:
            continue
        if url in processed:
            continue
        if url not in url_map:
            url_map[url] = {"record_ids":[rid], "fields": f}
        else:
            url_map[url]["record_ids"].append(rid)

    pending_urls = list(url_map.keys())
    print(f"Pending (unprocessed) unique URLs: {len(pending_urls)}")
    if not pending_urls:
        print("Nothing to do.")
        return pd.DataFrame([])

    to_process = pending_urls[:limit]
    print(f"Will process up to {len(to_process)} URLs (limit={limit}).")

    results = []
    method1_rows, method2_rows = [], []

    for url in tqdm(to_process):
        domain = get_domain(url)
        text, ctype = extract_text(url)
        if not text or len(text) < 300:
            # Skip too-short pages
            continue

        meta_fields   = url_map[url]["fields"]
        sent_at_val   = meta_fields.get("sent_at")
        extracted_val = meta_fields.get("extracted_at")

        # ===== 1) LLM → title / summary / key_points =====
        title, summary, key_points = "", "", []
        if strategy in ["llm_classification", "compare_all_methods", "hybrid_llm_title_keyword_tags"]:
            try:
                data = chat_json([system_prompt(), build_user_msg(url, text, domain)])
                title   = (data.get("title","") or "")[:500]
                summary = data.get("summary","") or ""
                key_points = (data.get("key_points", []) or [])
            except Exception as e:
                print(f"[WARN] LLM failed for {url}: {e}")
                title = ""
                summary = ""
                key_points = []

            # Fallback
            if not summary:
                preview = text[:400].replace("\n", " ").strip()

                summary = (
                    "[LLM-FALLBACK] Summary unavailable — using extracted page text.\n"
                    "(First 400 characters from the original page.)\n\n"
                    f"{preview}"
                )

                if not title:
                    title = (
                        f"[LLM-FALLBACK] {domain} | "
                        f"{url[:50]}"
                    )

            print("LLM summary length:", len(summary))  # debug


        # ===== 2) Tags: decide per strategy =====
        if strategy == "keyword_based":
            tags = method1_tags_from_text(text, url, ctype, topk=topk, topic_top1=topic_top1)
        elif strategy == "llm_classification":
            tags = method2_tags_llm(text, url, domain, ctype)
        elif strategy == "embedding_distance":
            tags = method3_tags_embeddings(text, url, ctype)
        elif strategy == "compare_all_methods":
            t1 = method1_tags_from_text(text, url, ctype, topk=topk, topic_top1=topic_top1)
            t2 = method2_tags_llm(text, url, domain, ctype)
            method1_rows.append({
                "url": url, "Resource": t1["Resource"], "Concept": t1["Concept"],
                "Topic": t1["Topic"], "Sub-Topic": t1["Sub-Topic"]
            })
            method2_rows.append({
                "url": url, "Resource": t2["Resource"], "Concept": t2["Concept"],
                "Topic": t2["Topic"], "Sub-Topic": t2["Sub-Topic"]
            })
            tags = t1
        elif strategy == "hybrid_llm_title_keyword_tags":
            # <<< Hybrid: keyword-based tags + LLM title/summary/key_points
            tags = method1_tags_from_text(text, url, ctype, topk=topk, topic_top1=topic_top1)
        else:
            tags = ensure_minimum_tags({"Resource":[], "Concept":[], "Topic":[], "Sub-Topic":[]}, ctype, url)

        row = {
            "url": url,
            "source_domain": domain,
            "title": title,
            "summary": summary,
            "key_points": "\n".join(key_points),
            "record_ids": ",".join(url_map[url]["record_ids"]),
            "sent_at":     sent_at_val,
            "extracted_at": extracted_val,
            "resource": tags.get("Resource", []),
            "concept":  tags.get("Concept",  []),
            "topic":    tags.get("Topic",    []),
            "subtopic": tags.get("Sub-Topic",[])
        }
        results.append(row)

    if not results:
        print("No results to write (all skipped as too short?).")
        return pd.DataFrame([])

    if write_out:
        write_records_existing_table(NEW_TABLE_NAME, results, batch_size=10)

    df = pd.DataFrame(results)
    ts = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    out_csv = f"/content/airtable_llm_summary_{strategy}_{ts}.csv"
    df.to_csv(out_csv, index=False)
    print("Saved CSV:", out_csv)

    if strategy == "compare_all_methods":
        stats = []
        stats.append(compute_stats(method1_rows, label="keyword_based", defaults=DEFAULT_PLACEHOLDERS))
        stats.append(compute_stats(method2_rows, label="llm_classification", defaults=DEFAULT_PLACEHOLDERS))
        df_stats = pd.DataFrame(stats)
        print("\n=== Tagging Stats (exclude placeholders) ===")
        print(df_stats.to_string(index=False))
        ts = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        stats_csv = f"/content/airtable_llm_summary_stats_{ts}.csv"
        df_stats.to_csv(stats_csv, index=False)
        print("Saved stats CSV:", stats_csv)

    print(f"Done. Rows prepared: {len(results)} | Written: {len(results) if write_out else 0}")
    return df

# ===== Run =====
df_result = process_with_strategy(
    limit=LIMIT,
    strategy=TAG_STRATEGY,
    write_out=WRITE_TO_AIRTABLE,
    topk=TOP_K_PER_FIELD,
    topic_top1=TOPIC_TOP1
)
df_result.head(10)
